Import Configured Workshop Secrets

In [ ]:
# Load your NGC API KEY here. You set this in the Secrets Manager in a previous step.
from dotenv import load_dotenv
_ = load_dotenv("../variables.env")
_ = load_dotenv("../secrets.env")

# Set your NGC org here. What org did you generate your NVIDIA key under?
import os
os.environ['NGC_CLI_ORG'] = "TODO_YOUR_NGC_ORG_HERE"

Install NGC CLI Tool

In [ ]:
%%capture
!sudo apt-get install unzip
!cd $HOME && wget --content-disposition https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/versions/4.8.2/files/ngccli_linux.zip -O ngccli_linux.zip && unzip ngccli_linux.zip && chmod u+x ngc-cli/ngc && echo "export PATH=\"\$PATH:$(pwd)/ngc-cli\"" >> ~/.bash_profile && source ~/.bash_profile && cd -

Docker login

In [ ]:
!docker login nvcr.io -u '$oauthtoken' -p $NGC_CLI_API_KEY

Download NeMo Microservices

In [ ]:
!/home/workbench/ngc-cli/ngc registry resource download-version "nvidia/nemo-microservices/nemo-microservices-quickstart:25.10"

Start NeMo Evaluator

In [ ]:
%%capture

os.environ['NEMO_MICROSERVICES_IMAGE_REGISTRY'] = "nvcr.io/nvidia/nemo-microservices"
os.environ['NEMO_MICROSERVICES_IMAGE_TAG'] = "25.10"

!cd nemo-microservices-quickstart_v25.10 && docker compose --profile evaluator up --detach --quiet-pull --wait && cd -

In [ ]:
import os, subprocess
import socket

# Try to get the gateway from /proc/net/route (Linux only)
with open('/proc/net/route') as f:
    for line in f:
        fields = line.strip().split()
        if fields[1] == '00000000':  # Default route
            gateway_hex = fields[2]
            # Convert hex to IP
            gateway_ip = socket.inet_ntoa(bytes.fromhex(gateway_hex)[::-1])
            print(f"Gateway IP: {gateway_ip}")
            break

os.environ["EVALUATOR_BASE_URL"] = f"http://{gateway_ip}:8080"
os.environ["NEMO_DATASTORE_URL"] = f"http://{gateway_ip}:3000"

# Namespace and dataset name used for hf:// URL
NAMESPACE = "default"
DATASET_NAME = "my-repo"

In [ ]:
import requests
import pandas as pd

# Download the HelpSteer2 dataset from Hugging Face
df = pd.read_json("hf://datasets/nvidia/HelpSteer2/train.jsonl.gz", lines=True)

# Extract only the prompt and response columns for evaluation
df = df[["prompt", "response"]].head(30)

# Save to a local file
file_name = "helpsteer2.jsonl"
df.to_json(file_name, orient="records", lines=True)

print(f"Dataset prepared with {len(df)} samples")
print(f"Sample data:")
print(df.head())

In [ ]:
import os
from huggingface_hub import HfApi

HF_ENDPOINT = f"{os.environ['NEMO_DATASTORE_URL']}/v1/hf"

hf_api = HfApi(endpoint=HF_ENDPOINT, token=os.environ["HF_TOKEN"])
repo_id = f"{NAMESPACE}/{DATASET_NAME}"

# Create the dataset repo if it doesn't exist
hf_api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

# Upload the file
result = hf_api.upload_file(
    path_or_fileobj=file_name,
    path_in_repo=file_name,
    repo_id=repo_id,
    repo_type="dataset",
    revision="main",
    commit_message=f"Eval dataset in {repo_id}"
)

print(f"Dataset uploaded: {result}")

In [ ]:
import os
from nemo_microservices import NeMoMicroservices

client = NeMoMicroservices(
    base_url=os.environ["EVALUATOR_BASE_URL"],
    timeout=600.0  # 10 minutes instead of default 60 seconds
)

# Model endpoint settings for the judge
MODEL_BASE_URL = "https://integrate.api.nvidia.com/v1"  # Add the url of your endpoint; it can be any model endpoint, such as an OpenAI endpoint, or a NIM endpoint.
MODEL_ID = "nvidia/llama-3.3-nemotron-super-49b-v1.5"                         # replace as needed

files_url = f"hf://datasets/{NAMESPACE}/{DATASET_NAME}"

# Inline config mirrors the developer notebook
config = {
    "type": "custom",
    "name": "my-config",
    "namespace": NAMESPACE,
    "params": {
        "limit_samples": 1,  # Only evaluate top row for testing
        "parallelism": 1,    # Reduce concurrent requests if hitting rate limits
        "request_timeout": 120,  # 2 minutes per request
        "max_retries": 3     # Retry failed requests
    },
    "tasks": {
        "my-task": {
            "type": "data",
            "metrics": {
                "my_eval": {
                    "type": "llm-judge",
                    "params": {
                        "model": {
                            "api_endpoint": {
                                "url": MODEL_BASE_URL,
                                "model_id": MODEL_ID,
                                "format": "openai",
                                "api_key": os.environ["NVIDIA_NIM_API_KEY"]
                            },
                            "prompt": {
                                "inference_params": {
                                    "temperature": 0.1,
                                    "max_tokens": 1024,
                                    "max_retries": 10,
                                    "request_timeout": 120  # THIS IS THE KEY - timeout for judge API calls
                                }
                            }
                        },
                        "template": {
                            "messages": [
                                {"role": "system", "content": "You are an expert evaluator for answers to user queries. Your task is to assess responses to user queries based on helpfulness, relevance, accuracy, and clarity."},
                                {"role": "user", "content": "Calculate the following metrics for the response: User Query: {{item.prompt}} Model Response: {{item.response}} Metrics: 1. Helpfulness (0-4): How well does the response help the user? 2. Correctness (0-4): Is the information correct? 3. Coherence (0-4): Is the response logically consistent and well-structured? 4. Complexity (0-4): How sophisticated is the response? 5. Verbosity (0-4): Is the response appropriately detailed? Instructions: Assign a score from 0 (poor) to 4 (excellent) for each metric. Respond in JSON format only: { \"helpfulness\": ..., \"correctness\": ..., \"coherence\": ..., \"complexity\": ..., \"verbosity\": ... }"}
                            ]
                        },
                        "scores": {
                            "helpfulness": {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"helpfulness\": *(\\d+)"}},
                            "correctness": {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"correctness\": *(\\d+)"}},
                            "coherence":   {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"coherence\": *(\\d+)"}},
                            "complexity":  {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"complexity\": *(\\d+)"}},
                            "verbosity":   {"type": "int", "parser": {"type": "regex", "pattern": "(?s).*\"verbosity\": *(\\d+)"}}
                        }
                    }
                }
            },
            "dataset": {"files_url": files_url}
        }
    }
}

target = {"type": "dataset", "dataset": {"files_url": files_url}}

job = client.evaluation.live(
    namespace=NAMESPACE,
    target=target,
    config=config,
    timeout=600.0  # Also override per-request
)

In [ ]:
# Access the results
print(f"Status: {job.status}\n")
print(f"Results: {job.result}\n")
print(f"Scores: {job.result.tasks["my-task"].metrics["my_eval"].scores}")

In [ ]:
import json

# Save results to file
results_dict = {
    "status": job.status,
    "result": job.result.model_dump(mode='json') if hasattr(job.result, 'model_dump') else job.result
}

with open("evaluation_results.json", "w") as f:
    json.dump(results_dict, f, indent=2, default=str)  # default=str as fallback

print("Saved to evaluation_results.json")

# If you want to access specific scores:
if job.result and job.result.tasks:
    for task_name, task_result in job.result.tasks.items():
        print(f"\nTask: {task_name}")
        if task_result.metrics:
            for metric_name, metric_result in task_result.metrics.items():
                print(f"  Metric: {metric_name}")
                if hasattr(metric_result, 'scores'):
                    print(f"  Scores: {metric_result.scores}")

In [ ]:
# Run multiple batches. Limiting to 1 sample per batch due to rate limits. 
batch_size = 1
all_results = []

for batch_num in range(0, 20, batch_size):  # Adjust total samples as needed
    print(f"Processing batch starting at sample {batch_num}...")
    
    # Add limit and offset to config for this batch
    batch_config = config.copy()
    batch_config["params"]["limit_samples"] = batch_size
    batch_config["params"]["offset"] = batch_num
    
    target = {"type": "dataset", "dataset": {"files_url": files_url}}
    
    try:
        response = client.evaluation.live(
            namespace=NAMESPACE,
            target=target,
            config=batch_config,
            timeout=600.0
        )
        
        all_results.append(response.result.model_dump(mode='json'))
        print(f"✓ Batch {batch_num} completed")
        
    except Exception as e:
        print(f"✗ Batch {batch_num} failed: {e}")
        continue

# Save combined results
with open("evaluation_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"Completed {len(all_results)} batches, saved to evaluation_results.json")